In [0]:
%pip install seaborn sqlalchemy

In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

sns.set_theme(style="whitegrid")
print("Librerías cargadas correctamente")

# Parte 1: Extracción y Diagnóstico (Extract)
En esta sección se realiza la extracción de los datos crudos desde el repositorio fuente mediante control de versiones (`git clone`), se leen los catálogos y se efectúa el diagnóstico exploratorio inicial (Data Profiling).

### 1.1 Descarga y Extracción de Fuentes de Datos
Se realiza la clonación del repositorio fuente mediante `git clone` para obtener los archivos crudos en el entorno local de trabajo.

In [0]:
# Comando para extraer/clonar los archivos crudos al entorno
!git clone https://github.com/IanCN-23/dalatam_ed3_TPI_Python.git

### 1.2 Carga de Archivos CSV en DataFrames de Pandas
Se carga el catálogo maestro de restaurantes y se unifican los 10 archivos divididos de menús en un único DataFrame (`df_menus`).

In [0]:
# Carga de restaurantes
df_restaurantes = pd.read_csv("dalatam_ed3_TPI_Python/restaurants.csv/restaurants.csv")

# Carga y consolidación de las 10 partes de menús
lista_menus = []
for i in range(1, 11):
    ruta = f"dalatam_ed3_TPI_Python/restaurant-menus.csv/restaurant_menus_parte_{i}.csv"
    df_parte = pd.read_csv(ruta)
    lista_menus.append(df_parte)

df_menus = pd.concat(lista_menus, ignore_index=True)
print("Archivos cargados exitosamente en Pandas.")

### 1.3 Data Profiling: Exploración Inicial de Datos
Se inspeccionan las dimensiones (`.shape`), los tipos de datos (`.dtypes`) y los valores faltantes (`.isnull().sum()`) de ambos conjuntos de datos.

In [0]:
# Diagnóstico completo de Menús
print("=== DIAGNÓSTICO MENÚS ===")
print("Dimensiones (.shape):", df_menus.shape)
print("\nTipos de Datos (.dtypes):\n", df_menus.dtypes)
print("\nValores Nulos por Columna:\n", df_menus.isnull().sum())

In [0]:
# Diagnóstico completo de Restaurantes
print("=== DIAGNÓSTICO RESTAURANTES ===")
print("Dimensiones (.shape):", df_restaurantes.shape)
print("\nTipos de Datos (.dtypes):\n", df_restaurantes.dtypes)
print("\nValores Nulos por Columna:\n", df_restaurantes.isnull().sum())

### 1.4 Inspección de Columnas y Muestra de Datos (`.head()`)
Se verifica la estructura formal de los encabezados de las tablas `df_restaurantes` y `df_menus`.

In [0]:
print("=== COLUMNAS DE MENÚS ===")
print(df_menus.columns.tolist())

print("\n--- Muestra (primeros 3 registros de Menús) ---")
display(df_menus.head(3))

In [0]:
print("=== COLUMNAS DE RESTAURANTES ===")
print(df_restaurantes.columns.tolist())
print("\n--- Muestra (primeros 3 registros de Restaurantes) ---")
display(df_restaurantes.head(3))

# Parte 2: Ingeniería de Características y Limpieza (Transform)
En esta etapa se procesan, limpian y estandarizan ambas fuentes de datos para asegurar su calidad antes de realizar la integración relacional.
### 2.1 Limpieza y Mapeo en Restaurantes (`df_restaurantes`)

In [0]:
# Mapeo de rangos de precio a texto descriptivo
diccionario_precios = {
    '$': 'Económico',     '$$': 'Moderadamente caro',     '$$$': 'Caro',     '$$$$': 'Muy caro'
}
df_restaurantes['rango_de_precios'] = df_restaurantes['price_range'].map(diccionario_precios)
# Muestra de la tabla de restaurantes procesada
display(df_restaurantes.head(2))

In [0]:
# Desanidado de la dirección completa (full_address)
split_dir = df_restaurantes['full_address'].str.split(',', expand=True)
df_restaurantes['calle'] = split_dir[0].str.strip()
df_restaurantes['ciudad'] = split_dir[1].str.strip()
df_restaurantes['estado_zip'] = split_dir[2].str.strip()
# Muestra de la tabla de restaurantes procesada
display(df_restaurantes.head(2))

In [0]:
# Filtro de calidad: puntajes válidos y minúsculas en categoría
df_restaurantes = df_restaurantes[
    (df_restaurantes['score'].notnull()) & (df_restaurantes['score'] > 0)
]
df_restaurantes['category'] = df_restaurantes['category'].astype(str).str.lower()

# Muestra de la tabla de restaurantes procesada
display(df_restaurantes.head(2))

### 2.2 Limpieza en `df_menus` e Integración (`df_master`)
Se realiza la conversión de la columna de precios a tipo numérico (`float`), se filtran los restaurantes consolidados (más de 100 valoraciones) y se unifican ambas tablas mediante un cruce relacional (`INNER JOIN`).

In [0]:
# Cruce relacional INNER JOIN
df_master = pd.merge(
    df_restaurantes_filtrado,
    df_menus,
    left_on='id',
    right_on='restaurant_id', # dado que los nombres de cada columna son diferentes pero refieren al mismo concepto
    how='inner'
)

print("Dimensiones finales de la tabla unificada df_master:", df_master.shape)
display(df_master.head(2))

In [0]:
# Limpieza y conversión de la columna de precios en menús
df_menus['price_clean'] = (
    df_menus['price']
    .astype(str)
    .str.replace('USD', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_menus['price_clean'] = pd.to_numeric(df_menus['price_clean'], errors='coerce')
df_menus = df_menus[(df_menus['price_clean'].notnull()) & (df_menus['price_clean'] > 0)]
display(df_master.tail(5))

In [0]:
# Filtro de restaurantes consolidados (> 100 valoraciones)
df_restaurantes_filtrado = df_restaurantes[df_restaurantes['ratings'] > 100]
display(df_master.head(2))

# Parte 3: Carga a Base de Datos Relacional (Load)
Se efectúa la persistencia del DataFrame unificado `df_master` en una base de datos relacional local SQLite denominada `delivery_insights.db`, creando la tabla `master_food_data` mediante SQLAlchemy.

In [0]:
# Creación del motor de conexión a SQLite y carga de datos
engine = create_engine('sqlite:///delivery_insights.db')

# Carga de df_master a la tabla master_food_data
df_master.to_sql('master_food_data', con=engine, if_exists='replace', index=False)

print("Base de datos delivery_insights.db y tabla master_food_data creadas exitosamente.")

# Parte 4: Business Intelligence y Reporte al CEO
En esta sección se realizan las consultas SQL a la base de datos SQLite y se generan los gráficos para el reporte ejecutivo.

### Pregunta 1: Penetración Geográfica
¿Cuáles son las 5 ciudades con mayor cantidad de restaurantes consolidados?

In [0]:
# Conexión a la base de datos SQLite
conn = engine.connect()

# Consulta SQL: Top 5 ciudades con más restaurantes
query_1 = """
SELECT ciudad, COUNT(DISTINCT id) AS total_restaurantes
FROM master_food_data
WHERE ciudad IS NOT NULL AND ciudad != ''
GROUP BY ciudad
ORDER BY total_restaurantes DESC
LIMIT 5;
"""

# Se guarda el resultado en un DataFrame y se muestra
df_q1 = pd.read_sql(text(query_1), conn)
display(df_q1)

In [0]:
# Gráfico de barras horizontal ordenado
plt.figure(figsize=(8, 4))
sns.barplot(data=df_q1, x='total_restaurantes', y='ciudad', palette='Blues_r')
plt.title('Top 5 Ciudades con Mayor Cantidad de Restaurantes')
plt.xlabel('Cantidad de Restaurantes Consolidados')
plt.ylabel('Ciudad')
plt.show()

> **Conclusión:** Se observa una mayor concentración de restaurantes en Milwaukee (122) y Seattle (119), que representan la mayor parte de los locales consolidados. Para aprovechar mejor los recursos y el tiempo, conviene empezar las operaciones principalmente en estas dos ciudades.

### Pregunta 2: Análisis de Oportunidad de Precio
Se evalúa la distribución de los restaurantes según su categoría de precio para determinar si el mercado está dominado por opciones costosas.

In [0]:
# Consulta SQL: Contar restaurantes únicos por rango de precio
query_2 = """
SELECT rango_de_precios, COUNT(DISTINCT id) AS total_restaurantes
FROM master_food_data
WHERE rango_de_precios IS NOT NULL AND rango_de_precios != ''
GROUP BY rango_de_precios
ORDER BY total_restaurantes DESC;
"""

# Ejecución del query y muestra de resultados
df_q2 = pd.read_sql(text(query_2), conn)
display(df_q2)

In [0]:
# Gráfico de torta (Pie Chart) para la distribución por precio
plt.figure(figsize=(6, 6))
plt.pie(
    df_q2['total_restaurantes'], 
    labels=df_q2['rango_de_precios'], 
    autopct='%1.1f%%', 
    startangle=90
)
plt.title('Distribución de Restaurantes por Rango de Precios')
plt.show()

> **Conclusión:** El gráfico muestra que el 99.3% de los restaurantes está en las categorías Económico (73.6%) y Moderadamente caro (25.7%), mientras que la categoría Caro solo representa el 0.7%. Esto demuestra que la gran mayoría de la oferta maneja precios accesibles y orientados al consumo masivo.

### Pregunta 3: Precio Promedio del Menú en las Top 3 Ciudades
Se calcula el precio promedio de los platos del menú en las tres ciudades con más restaurantes (Milwaukee, Seattle y Lynnwood) para comparar en cuál resulta más caro comer.

In [0]:
# Consulta SQL: Precio promedio en las 3 ciudades con más restaurantes
query_3 = """
SELECT ciudad, ROUND(AVG(price_clean), 2) AS precio_promedio
FROM master_food_data
WHERE ciudad IN (
    SELECT ciudad
    FROM master_food_data
    WHERE ciudad IS NOT NULL AND ciudad != ''
    GROUP BY ciudad
    ORDER BY COUNT(DISTINCT id) DESC
    LIMIT 3
)
GROUP BY ciudad
ORDER BY precio_promedio DESC;
"""

# Ejecución del query y muestra de la tabla
df_q3 = pd.read_sql(text(query_3), conn)
display(df_q3)

In [0]:
# Gráfico de barras comparativo de precio promedio
plt.figure(figsize=(7, 4))
sns.barplot(data=df_q3, x='ciudad', y='precio_promedio', palette='Greens_r')
plt.title('Precio Promedio por Plato en las Top 3 Ciudades')
plt.xlabel('Ciudad')
plt.ylabel('Precio Promedio (USD)')
plt.show()

> **Conclusión:** Al analizar los precios promedio, Seattle presenta el plato más caro con $11.98 USD, seguido de Lynnwood ($10.82 USD) y Milwaukee ($9.53 USD). Aunque Milwaukee tiene una mayor cantidad de restaurantes, Seattle muestra un valor por plato más elevado, lo que permite proyectar un ticket de venta más alto por pedido.

### Pregunta 4: Relación entre Rango de Precios y Calificación (Score)
Se evalúa si los restaurantes pertenecientes a categorías de precios más altas obtienen una mejor calificación promedio por parte de los clientes.

In [0]:
# Consulta SQL: Puntaje promedio según el rango de precios
query_4 = """
SELECT rango_de_precios, ROUND(AVG(score), 2) AS score_promedio
FROM master_food_data
WHERE rango_de_precios IS NOT NULL AND rango_de_precios != ''
GROUP BY rango_de_precios
ORDER BY score_promedio DESC;
"""

# Ejecución del query y muestra de la tabla
df_q4 = pd.read_sql(text(query_4), conn)
display(df_q4)

In [0]:
# Gráfico de líneas para observar la tendencia
plt.figure(figsize=(7, 4))
sns.lineplot(data=df_q4, x='rango_de_precios', y='score_promedio', marker='o', color='red', linewidth=2)
plt.title('Calificación Promedio según Rango de Precios')
plt.xlabel('Rango de Precios')
plt.ylabel('Puntaje Promedio (0 a 5)')
plt.ylim(0, 5)
plt.show()

> **Conclusión:** Se observa que las calificaciones promedio son bastante parecidas entre categorías. Los restaurantes de nivel "Moderadamente caro" (4.62) y "Económico" (4.61) tienen prácticamente la misma nota, mientras que los de nivel "Caro" bajan a 4.43. Esto demuestra que cobrar más caro no garantiza recibir una mejor puntuación por parte de los clientes.

# Conclusiones finales y recomendaciones

A partir del análisis realizado en el proyecto, se destacan los siguientes puntos principales:

- Se recomienda priorizar el inicio de operaciones en Milwaukee y Seattle por concentrar la mayor cantidad de locales consolidados.
- La oferta disponible se enfoca principalmente en el consumo masivo, dado que el 99.3% de los locales pertenece a las categorías de precio económico y moderado.
- Aunque Milwaukee cuenta con una mayor cantidad de restaurantes, Seattle registra un precio promedio por plato más elevado ($11.98 USD), lo que sugiere un ticket promedio superior por pedido.
- La puntuación atribuida por los usuarios se mantiene constante (~4.6 estrellas) en los distintos niveles de precio, lo que indica que el valor del plato no condiciona la percepción de calidad del cliente.